In [2]:
from pathlib import Path
import os, shutil
import pandas as pd

DATA = Path(r"C:\data\brats")
out = Path(r"C:\data\nnUNet_raw\Dataset501_BraTS2023GLI")
df = pd.read_csv(r"..\splits\index.csv")
MODS = {"t1n": "0000", "t1c": "0001", "t2w": "0002", "t2f": "0003"}

def place(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

print(len(df), "cases in index")

1346 cases in index


In [3]:
import pandas as pd
DATA = Path(r"C:\data\brats")
df = pd.read_csv(r"..\splits\index.csv")
MODS = {"t1n": "0000", "t1c": "0001", "t2w": "0002", "t2f": "0003"}

def convert_row(row, img_dir, lbl_dir):
    cid = row["case_id"]
    for m, num in MODS.items():
        place(DATA / row[m], out / img_dir / f"{cid}_{num}.nii.gz")
    place(DATA / row["seg"], out / lbl_dir / f"{cid}.nii.gz")

row = df[df.split == "external_test"].iloc[0]
convert_row(row, "imagesTs_africa", "labelsTs_africa")
print(sorted(p.name for p in (out / "imagesTs_africa").iterdir()))

['BraTS-SSA-00002-000_0000.nii.gz', 'BraTS-SSA-00002-000_0001.nii.gz', 'BraTS-SSA-00002-000_0002.nii.gz', 'BraTS-SSA-00002-000_0003.nii.gz']


In [4]:
FOLDERS = {
    "train":         ("imagesTr", "labelsTr"),
    "val":           ("imagesTr", "labelsTr"),
    "test":          ("imagesTs_gli_test", "labelsTs_gli_test"),
    "external_test": ("imagesTs_africa", "labelsTs_africa"),
}

for i, row in enumerate(df.itertuples(index=False)):
    img_dir, lbl_dir = FOLDERS[row.split]
    convert_row(row._asdict(), img_dir, lbl_dir)
    if i % 100 == 0:
        print(i, row.case_id)
print("done")

0 BraTS-GLI-00000-000
100 BraTS-GLI-00105-000
200 BraTS-GLI-00241-000
300 BraTS-GLI-00383-000
400 BraTS-GLI-00510-001
500 BraTS-GLI-00610-001
600 BraTS-GLI-00723-000
700 BraTS-GLI-00999-000
800 BraTS-GLI-01099-000
900 BraTS-GLI-01198-000
1000 BraTS-GLI-01298-000
1100 BraTS-GLI-01398-000
1200 BraTS-GLI-01498-000
1300 BraTS-SSA-00132-000
done


In [5]:
for d in sorted(out.iterdir()):
    if d.is_dir():
        print(f"{d.name:<22} {len(list(d.glob('*.nii.gz')))}")

imagesTr               4244
imagesTs_africa        380
imagesTs_gli_test      760
labelsTr               1061
labelsTs_africa        95
labelsTs_gli_test      190


In [6]:
import json

dataset = {
    "channel_names": {"0": "T1", "1": "T1c", "2": "T2", "3": "FLAIR"},
    "labels": {"background": 0,
               "whole_tumor": [1, 2, 3],
               "tumor_core": [1, 3],
               "enhancing_tumor": [3]},
    "regions_class_order": [2, 1, 3],
    "numTraining": len(list((out / "labelsTr").glob("*.nii.gz"))),
    "file_ending": ".nii.gz",
}
(out / "dataset.json").write_text(json.dumps(dataset, indent=2))
print(dataset["numTraining"])

1061


In [7]:
train_ids = df[df.split == "train"].case_id.tolist()
val_ids   = df[df.split == "val"].case_id.tolist()
splits = [{"train": train_ids, "val": val_ids}]
(out / "splits_final.json").write_text(json.dumps(splits, indent=2))
print(len(train_ids), len(val_ids))

869 192


In [8]:
repo_nn = Path(r"..\nnunet"); repo_nn.mkdir(exist_ok=True)
for f in ["dataset.json", "splits_final.json"]:
    shutil.copy2(out / f, repo_nn / f)